# PubMed corpus collection and LLM retrieval refinement

This notebook collects recall-oriented PubMed candidates and uses a
pathogen-specific LLM refinement pass to remove abstracts that are not
actually about the target pathogen. It writes a filtered
`refined_corpus_articles.parquet` for the category notebook. All outputs are
local, resumable Parquet checkpoints under `outputs/pubmed_screening/`.

In [1]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from graphicalizer import (
    OpenAIChatCompleter,
    PathogenSpec,
    PubMedClient,
    build_pathogen_search_query,
    collect_pubmed_corpus,
    load_refinement_run,
    load_search_bundle,
    refine_pubmed_corpus,
)

## Configure pathogens, YAML search bundles, and checkpoints

In [2]:
PUBMED_EMAIL = os.environ.get('PUBMED_EMAIL', 'xfcosta@gmail.com')

PATHOGENS = {
    'Coxiella burnetii': ['Coxiella', 'C. burnetii'],
}
SEARCH_BUNDLE_DIR = PROJECT_ROOT / 'assets' / 'search_bundles'
SEARCH_BUNDLES = {
    bundle.search_type: bundle
    for bundle in (
        load_search_bundle(SEARCH_BUNDLE_DIR / 'pubmed_animal_evidence.yaml', expected_search_type='animal'),
        load_search_bundle(SEARCH_BUNDLE_DIR / 'pubmed_zoonosis_evidence.yaml', expected_search_type='zoonosis'),
    )
}

PUBMED_MAX_RESULTS_PER_QUERY = 5000
PUBMED_SEARCH_PAGE_SIZE = 1000
PUBMED_FETCH_BATCH_SIZE = 200
RESUME = True
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'pubmed_screening'

OPENAI_MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
LLM_MAX_TOKENS = 1024
LLM_RETRIES = 3
LLM_MAX_CALLS = None
LLM_SAVE_EVERY = 10
RETRY_FAILED_LLM_ROWS = False

pubmed = PubMedClient(
    email=PUBMED_EMAIL,
    api_key=os.environ.get('PUBMED_API_KEY'),
    tool='recursive-framing-graphicalizer',
    retries=3,
)


## Preview the generated queries

In [3]:
for pathogen, aliases in PATHOGENS.items():
    spec = PathogenSpec(pathogen, tuple(aliases))
    print(f'--- {pathogen} / animal ---')
    print(build_pathogen_search_query(spec, 'animal', search_bundles=SEARCH_BUNDLES))
    print(f'--- {pathogen} / zoonosis ---')
    print(build_pathogen_search_query(spec, 'zoonosis', search_bundles=SEARCH_BUNDLES))

--- Coxiella burnetii / animal ---
("Coxiella burnetii"[Title/Abstract] OR Coxiella[Title/Abstract] OR "C. burnetii"[Title/Abstract]) AND (animal[Title/Abstract] OR host[Title/Abstract] OR wildlife[Title/Abstract] OR livestock[Title/Abstract] OR veterinary[Title/Abstract] OR infection[Title/Abstract])
--- Coxiella burnetii / zoonosis ---
("Coxiella burnetii"[Title/Abstract] OR Coxiella[Title/Abstract] OR "C. burnetii"[Title/Abstract]) AND (zoonosis[Title/Abstract] OR zoonotic[Title/Abstract] OR spillover[Title/Abstract] OR animal-to-human[Title/Abstract] OR "human infection"[Title/Abstract] OR "human disease"[Title/Abstract] OR transmission[Title/Abstract])


## Collect and resume the deduplicated corpus

In [4]:
collection = collect_pubmed_corpus(
    pubmed,
    PATHOGENS,
    OUTPUT_DIR,
    search_bundles=SEARCH_BUNDLES,
    max_results_per_query=PUBMED_MAX_RESULTS_PER_QUERY,
    search_page_size=PUBMED_SEARCH_PAGE_SIZE,
    fetch_batch_size=PUBMED_FETCH_BATCH_SIZE,
    resume=RESUME,
)
corpus = collection.corpus
print('Corpus rows:', len(corpus))
print('Search failures:', collection.manifest['search_failures'])
print(corpus.groupby(['pathogen', 'fetch_status']).size())

Corpus rows: 3738
Search failures: 0
pathogen           fetch_status
Coxiella burnetii  no_abstract      180
                   not_returned       4
                   ok              3554
dtype: int64


## Refine retrieval with target-pathogen attribution

In [7]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set OPENAI_API_KEY before running the LLM refinement.')
llm = OpenAIChatCompleter(OPENAI_MODEL)
refinement_run = refine_pubmed_corpus(
    corpus,
    PATHOGENS,
    llm,
    OUTPUT_DIR,
    model=OPENAI_MODEL,
    max_tokens=LLM_MAX_TOKENS,
    retries=LLM_RETRIES,
    max_llm_calls=LLM_MAX_CALLS,
    save_every=LLM_SAVE_EVERY,
    resume=RESUME,
    retry_failed=RETRY_FAILED_LLM_ROWS,
)
print('Candidate rows:', len(corpus))
print('Refinement decisions:', len(refinement_run.refinement))
print('Retained rows:', len(refinement_run.refined_corpus))
print('Rejected/needs review rows:', max(0, len(refinement_run.refinement) - len(refinement_run.refined_corpus) - len(refinement_run.failures)))
print('Retryable failures:', len(refinement_run.failures))

Refined 1/3533: Coxiella burnetii / PMID 10530457
Refined 2/3533: Coxiella burnetii / PMID 10549097
Refined 3/3533: Coxiella burnetii / PMID 10574071
Refined 4/3533: Coxiella burnetii / PMID 10603382
Refined 5/3533: Coxiella burnetii / PMID 10608766
Refined 6/3533: Coxiella burnetii / PMID 10614951
Refined 7/3533: Coxiella burnetii / PMID 106503
Refined 8/3533: Coxiella burnetii / PMID 10699325
Refined 9/3533: Coxiella burnetii / PMID 10743344
Refined 10/3533: Coxiella burnetii / PMID 10757226
Refined 11/3533: Coxiella burnetii / PMID 10771709
Refined 12/3533: Coxiella burnetii / PMID 10783505
Refined 13/3533: Coxiella burnetii / PMID 10796960
Refined 14/3533: Coxiella burnetii / PMID 10817644
Refined 15/3533: Coxiella burnetii / PMID 10831714
Refined 16/3533: Coxiella burnetii / PMID 108342
Refined 17/3533: Coxiella burnetii / PMID 1085666
Refined 18/3533: Coxiella burnetii / PMID 10858189
Refined 19/3533: Coxiella burnetii / PMID 10918906
Refined 20/3533: Coxiella burnetii / PMID 109

KeyboardInterrupt: 

## Inspect refinement quality and provenance

Only rows with accepted target-pathogen attribution are written to
`refined_corpus_articles.parquet`. Rejected, ambiguous, and failed rows
remain in the refinement checkpoint for audit and retry; they are not
silently treated as biological negatives.

In [9]:
import pandas as pd

# Always reconstruct the displayed state from checkpoints so this cell
# also works after manually interrupting the refinement cell.
corpus_checkpoint = OUTPUT_DIR / 'corpus_articles.parquet'
if corpus_checkpoint.exists():
    corpus = pd.read_parquet(corpus_checkpoint)
elif 'corpus' not in globals():
    raise FileNotFoundError(f'No corpus checkpoint found at {corpus_checkpoint}.')
refinement_run = load_refinement_run(corpus, OUTPUT_DIR)

print('Output directory:', OUTPUT_DIR)
print('Files:', sorted(path.name for path in OUTPUT_DIR.glob('*') if path.is_file()))
display(refinement_run.refinement[[
    'pathogen', 'pmid', 'refinement_keep', 'accepted',
    'target_pathogen_supported', 'evidence_relevant', 'confidence',
    'review_required', 'rationale'
]])

Output directory: /Users/f.costa/Code/RecursiveFraming/outputs/pubmed_screening
Files: ['corpus_articles.parquet', 'corpus_articles.parquet.lock', 'llm_refinement.parquet', 'llm_refinement.parquet.lock', 'run_manifest.json', 'search_pmids.parquet', 'search_pmids.parquet.lock', 'search_runs.parquet', 'search_runs.parquet.lock']


,pathogen,pmid,refinement_keep,accepted,target_pathogen_supported,evidence_relevant,confidence,review_required,rationale
0,Coxiella burnetii,10052552,True,True,True,True,0.90,False,The abstract discusses a Q fever outbreak and ...
1,Coxiella burnetii,10190351,False,False,False,False,0.80,True,The abstract discusses infective endocarditis ...
2,Coxiella burnetii,10219646,True,True,True,True,1.00,False,The abstract discusses the treatment of granul...
3,Coxiella burnetii,10327998,True,True,True,True,0.90,False,The abstract discusses a case involving Coxiel...
4,Coxiella burnetii,10341175,True,True,True,True,0.90,False,The abstract discusses Q fever and its causati...
...,...,...,...,...,...,...,...,...,...
72,Coxiella burnetii,11734750,True,True,True,True,1.00,False,The abstract discusses a case of hemolytic-ure...
73,Coxiella burnetii,11769382,True,True,True,True,0.85,False,The abstract discusses the estimation of Q fev...
74,Coxiella burnetii,11774896,True,True,True,True,1.00,False,The abstract clearly discusses the isolation a...
75,Coxiella burnetii,11775317,True,True,True,True,0.90,False,The abstract discusses a case of acute Q fever...
